<a href="https://colab.research.google.com/github/Stellar-merge/TBO/blob/main/Hotel_cancellation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
import pandas as pd

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

In [ ]:
print("Dataset Shape:", df.shape)

print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

In [ ]:
display(df.head())

In [ ]:
print("Dataset Columns:")

for col in df.columns:
    print(col)

In [ ]:
missing = df.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
print(df["is_canceled"].value_counts())

print("\nCancellation Percentage:")
print(df["is_canceled"].value_counts(normalize=True) * 100)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
df = df.drop(
    columns=["reservation_status", "reservation_status_date"],
    errors="ignore"
)

print("Leakage columns removed!")

In [ ]:
df["children"] = pd.to_numeric(
    df["children"], errors="coerce"
).fillna(0)

df["country"] = df["country"].fillna("Unknown")

df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)

print("Missing values handled!")

In [ ]:
df = df[
    (df["adults"] + df["children"] + df["babies"]) > 0
]

df = df[
    (df["adr"] >= 0) &
    (df["adr"] < 1000)
]

print("Invalid records removed!")
print("Updated shape:", df.shape)

In [ ]:
print("Final Dataset Shape:", df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="is_canceled")

plt.title("Booking Cancellation Distribution")
plt.xlabel("Cancellation (0 = No, 1 = Yes)")
plt.ylabel("Number of Bookings")

plt.show()

In [ ]:
cancellation = df["is_canceled"].value_counts(normalize=True) * 100

print("Not Cancelled:", round(cancellation.get(0, 0), 2), "%")
print("Cancelled:", round(cancellation.get(1, 0), 2), "%")

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=df,
    x="hotel",
    y="is_canceled"
)

plt.title("Cancellation Rate by Hotel Type")
plt.ylabel("Cancellation Rate")
plt.xlabel("Hotel Type")

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=df,
    x="customer_type",
    y="is_canceled"
)

plt.title("Cancellation Rate by Customer Type")
plt.ylabel("Cancellation Rate")
plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=df,
    x="deposit_type",
    y="is_canceled"
)

plt.title("Cancellation Rate by Deposit Type")
plt.ylabel("Cancellation Rate")

plt.show()

In [ ]:
plt.figure(figsize=(15, 10))

numeric_df = df.select_dtypes(include=["number"])

sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
month_mapping = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

df["month_num"] = df["arrival_date_month"].map(month_mapping)

print(df[["arrival_date_month", "month_num"]].head())

In [ ]:
df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

In [ ]:
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

In [ ]:
df["room_changed"] = (
    df["reserved_room_type"] != df["assigned_room_type"]
).astype(int)

In [ ]:
df["has_company"] = (df["company"].notna()).astype(int)

df["has_agent"] = (df["agent"].notna()).astype(int)

In [ ]:
df["has_company"] = (
    pd.to_numeric(df["company"], errors="coerce").fillna(0) != 0
).astype(int)

df["has_agent"] = (
    pd.to_numeric(df["agent"], errors="coerce").fillna(0) != 0
).astype(int)

In [ ]:
df["total_cost"] = df["adr"] * df["total_nights"]

In [ ]:
booking_date = pd.to_datetime({
    "year": df["arrival_date_year"],
    "month": df["month_num"],
    "day": df["arrival_date_day_of_month"]
})

df["dow"] = booking_date.dt.dayofweek

In [ ]:
booking_date_actual = (
    booking_date -
    pd.to_timedelta(df["lead_time"], unit="D")
)

df["booking_ordinal"] = booking_date_actual.map(
    pd.Timestamp.toordinal
)

In [ ]:
df["day_of_year"] = booking_date.dt.dayofyear

In [ ]:
df["adr_per_guest"] = (
    df["adr"] / df["total_guests"].clip(lower=1)
)

In [ ]:
df["prev_cancel_ratio"] = (
    df["previous_cancellations"] /
    (
        df["previous_cancellations"] +
        df["previous_bookings_not_canceled"] + 1
    )
)

In [ ]:
df = df.drop(columns=["arrival_date_month"])

In [ ]:
import numpy as np

In [ ]:
new_features = [
    "month_num",
    "total_nights",
    "total_guests",
    "room_changed",
    "has_company",
    "has_agent",
    "total_cost",
    "dow",
    "day_of_year",
    "booking_ordinal",
    "adr_per_guest",
    "prev_cancel_ratio"
]

In [ ]:
import numpy as np

print("\nInfinite Values:")
print(np.isinf(df[new_features].select_dtypes(include="number")).sum())

In [ ]:
new_features = [
    "month_num",
    "total_nights",
    "total_guests",
    "room_changed",
    "has_company",
    "has_agent",
    "total_cost",
    "dow",
    "day_of_year",
    "booking_ordinal",
    "adr_per_guest",
    "prev_cancel_ratio"
]

print("New Features Created:")

for feature in new_features:
    print(feature, ":", feature in df.columns)

print("\nFinal Dataset Shape:", df.shape)

print("\nMissing Values in New Features:")
print(df[new_features].isnull().sum())

print("\nInfinite Values:")
print(np.isinf(df[new_features].select_dtypes(include="number")).sum())

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Categorical Columns:")
print(categorical_cols)

print("\nNumber of categorical columns:", len(categorical_cols))

In [ ]:
for col in categorical_cols:
    df[col] = df[col].astype("category")

print("Categorical columns converted successfully!")

print(df[categorical_cols].dtypes)

In [ ]:
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42
)

print("Training:", X_tr.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [ ]:
print("Training categorical columns:")

print(
    X_tr.select_dtypes(include=["category"]).columns.tolist()
)

In [ ]:
print("Training target distribution:")
print(y_tr.value_counts(normalize=True) * 100)

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
!pip install -q xgboost

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

print("XGBoost Version:", xgb.__version__)

In [ ]:
baseline_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

print("Baseline XGBoost model created!")

In [ ]:
import time

In [ ]:
start_time = time.time()

baseline_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=100
)

end_time = time.time()

print("Training completed!")
print("Training Time:", round(end_time - start_time, 2), "seconds")
print("Best Iteration:", baseline_model.best_iteration)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

val_pred = baseline_model.predict(X_val)
val_prob = baseline_model.predict_proba(X_val)[:, 1]

print("BASELINE XGBOOST RESULTS")

print("Accuracy:", round(accuracy_score(y_val, val_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_val, val_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_val, val_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_val, val_pred) * 100, 2), "%")
print("ROC-AUC:", round(roc_auc_score(y_val, val_prob), 4))

print("\nClassification Report:")
print(classification_report(y_val, val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

In [ ]:
plt.figure(figsize=(7, 5))

sns.heatmap(
    confusion_matrix(y_val, val_pred),
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("XGBoost Baseline Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

val_pred = baseline_model.predict(X_val)

accuracy = accuracy_score(y_val, val_pred) * 100

print(f"XGBoost Model Accuracy: {accuracy:.2f}%")

In [ ]:
test_pred = baseline_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred) * 100

print(f"XGBoost Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
!pip install -q xgboost scipy scikit-learn

import numpy as np
import time

from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

print("Tuning libraries ready!")

In [ ]:
param_dist = {
    "n_estimators": [500, 800, 1200],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.02, 0.03, 0.05, 0.07],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "gamma": [0, 0.1, 0.2, 0.3],
    "reg_alpha": [0, 0.01, 0.1, 0.5],
    "reg_lambda": [1, 2, 5, 10]
}

In [ ]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=15,
    scoring="accuracy",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=1,
    refit=True
)

print("Hyperparameter tuning configured!")

In [ ]:
start_time = time.time()

random_search.fit(X_tr, y_tr)

end_time = time.time()

print("\nTUNING COMPLETED!")

print(
    "Total Training Time:",
    round((end_time - start_time) / 60, 2),
    "minutes"
)

print("\nBEST CROSS-VALIDATION ACCURACY:")
print(round(random_search.best_score_ * 100, 2), "%")

print("\nBEST PARAMETERS:")
print(random_search.best_params_)

In [ ]:
best_params = random_search.best_params_.copy()

tuned_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=60,
    random_state=42,
    n_jobs=-1
)

tuned_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=100
)

print("Tuned XGBoost model trained!")
print("Best Iteration:", tuned_model.best_iteration)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

val_pred_tuned = tuned_model.predict(X_val)
val_prob_tuned = tuned_model.predict_proba(X_val)[:, 1]

print("\nTUNED XGBOOST RESULTS")

print("Accuracy:",
      round(accuracy_score(y_val, val_pred_tuned) * 100, 2), "%")

print("Precision:",
      round(precision_score(y_val, val_pred_tuned) * 100, 2), "%")

print("Recall:",
      round(recall_score(y_val, val_pred_tuned) * 100, 2), "%")

print("F1 Score:",
      round(f1_score(y_val, val_pred_tuned) * 100, 2), "%")

print("ROC-AUC:",
      round(roc_auc_score(y_val, val_prob_tuned), 4))

print("\nClassification Report:")
print(classification_report(y_val, val_pred_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_tuned))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get cancellation probabilities
val_prob = tuned_model.predict_proba(X_val)[:, 1]

results = []

# Test different classification thresholds
for threshold in np.arange(0.20, 0.81, 0.01):

    predictions = (val_prob >= threshold).astype(int)

    results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions),
        "f1": f1_score(y_val, predictions)
    })

threshold_results = pd.DataFrame(results)

# Sort by accuracy
threshold_results = threshold_results.sort_values(
    by="accuracy",
    ascending=False
)

print("TOP 10 THRESHOLD RESULTS")

display(threshold_results.head(10))

In [ ]:
best_threshold = threshold_results.iloc[0]["threshold"]

print("Best Threshold:", best_threshold)
print(
    "Best Validation Accuracy:",
    round(threshold_results.iloc[0]["accuracy"] * 100, 2),
    "%"
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    threshold_results["threshold"],
    threshold_results["accuracy"] * 100,
    marker="o"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Accuracy (%)")
plt.title("XGBoost Threshold Optimization")
plt.grid(True)

plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

optimized_pred = (val_prob >= best_threshold).astype(int)

print("OPTIMIZED XGBOOST RESULTS")

print(
    "Accuracy:",
    round(accuracy_score(y_val, optimized_pred) * 100, 2),
    "%"
)

print(
    "Precision:",
    round(precision_score(y_val, optimized_pred) * 100, 2),
    "%"
)

print(
    "Recall:",
    round(recall_score(y_val, optimized_pred) * 100, 2),
    "%"
)

print(
    "F1 Score:",
    round(f1_score(y_val, optimized_pred) * 100, 2),
    "%"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, optimized_pred))

print("\nClassification Report:")
print(classification_report(y_val, optimized_pred))

In [ ]:
print("Best Threshold:", best_threshold)

print("Best Accuracy:",
      round(threshold_results.iloc[0]["accuracy"] * 100, 2), "%")

print("Precision:",
      round(threshold_results.iloc[0]["precision"] * 100, 2), "%")

print("Recall:",
      round(threshold_results.iloc[0]["recall"] * 100, 2), "%")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

importance_df = pd.DataFrame({
    "Feature": X_tr.columns,
    "Importance": tuned_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("TOP 20 IMPORTANT FEATURES")

display(importance_df.head(20))

In [ ]:
plt.figure(figsize=(11, 8))

top_features = importance_df.head(20)

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 20 XGBoost Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

In [ ]:
print("LOW IMPORTANCE FEATURES")

display(
    importance_df.tail(15)
)

In [ ]:
# Features that may contain information unavailable at booking creation
booking_time_excluded = [
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "room_changed"
]

# Create booking-time dataset
booking_df = df.drop(
    columns=booking_time_excluded,
    errors="ignore"
).copy()

# Separate features and target
X_booking = booking_df.drop(columns=["is_canceled"])
y_booking = booking_df["is_canceled"]

print("Booking-time dataset shape:", X_booking.shape)
print("\nRemaining excluded columns:")
print([col for col in booking_time_excluded if col in X_booking.columns])

print("\nTarget distribution:")
print(y_booking.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% training pool, 20% test
X_train_pool, X_test_booking, y_train_pool, y_test_booking = train_test_split(
    X_booking,
    y_booking,
    test_size=0.20,
    random_state=42,
    stratify=y_booking
)

# Second split: training and validation
X_train_booking, X_val_booking, y_train_booking, y_val_booking = train_test_split(
    X_train_pool,
    y_train_pool,
    test_size=0.15,
    random_state=42,
    stratify=y_train_pool
)

print("Training:", X_train_booking.shape)
print("Validation:", X_val_booking.shape)
print("Testing:", X_test_booking.shape)

In [ ]:
from xgboost import XGBClassifier

booking_model = XGBClassifier(
    **tuned_model.get_params()
)

# Fit the new model
booking_model.fit(
    X_train_booking,
    y_train_booking,
    eval_set=[(X_val_booking, y_val_booking)],
    verbose=100
)

print("Booking-time XGBoost training completed!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Predict on validation data
y_val_pred = booking_model.predict(X_val_booking)
y_val_prob = booking_model.predict_proba(X_val_booking)[:, 1]

# Calculate metrics
print("BOOKING-TIME XGBOOST RESULTS")

print("Accuracy:", round(accuracy_score(y_val_booking, y_val_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_val_booking, y_val_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_val_booking, y_val_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_val_booking, y_val_pred) * 100, 2), "%")
print("ROC-AUC:", round(roc_auc_score(y_val_booking, y_val_prob), 4))

print("\nClassification Report:")
print(classification_report(y_val_booking, y_val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_booking, y_val_pred))

In [ ]:
print("BOOKING-TIME MODEL FEATURES")
for i, col in enumerate(X_booking.columns, 1):
    print(f"{i}. {col}")

print("\nTotal features:", len(X_booking.columns))

In [ ]:
# Potentially time-dependent features
exclude_features = [
    "required_car_parking_spaces",
    "total_of_special_requests",
    "adr",
    "total_cost",
    "adr_per_guest"
]

# Remove them from existing splits
X_train_booking_v2 = X_train_booking.drop(
    columns=exclude_features
)

X_val_booking_v2 = X_val_booking.drop(
    columns=exclude_features
)

X_test_booking_v2 = X_test_booking.drop(
    columns=exclude_features
)

print("New training shape:", X_train_booking_v2.shape)
print("New validation shape:", X_val_booking_v2.shape)
print("New testing shape:", X_test_booking_v2.shape)

In [ ]:
from xgboost import XGBClassifier

booking_model_v2 = XGBClassifier(
    **tuned_model.get_params()
)

booking_model_v2.fit(
    X_train_booking_v2,
    y_train_booking,
    eval_set=[(X_val_booking_v2, y_val_booking)],
    verbose=100
)

print("Stricter booking-time model training completed!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Validation predictions
y_pred_v2 = booking_model_v2.predict(X_val_booking_v2)
y_prob_v2 = booking_model_v2.predict_proba(X_val_booking_v2)[:, 1]

print("STRICT BOOKING-TIME XGBOOST RESULTS")

print("Accuracy:", round(accuracy_score(y_val_booking, y_pred_v2) * 100, 2), "%")
print("Precision:", round(precision_score(y_val_booking, y_pred_v2) * 100, 2), "%")
print("Recall:", round(recall_score(y_val_booking, y_pred_v2) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_val_booking, y_pred_v2) * 100, 2), "%")
print("ROC-AUC:", round(roc_auc_score(y_val_booking, y_prob_v2), 4))

print("\nClassification Report:")
print(classification_report(y_val_booking, y_pred_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_booking, y_pred_v2))

In [ ]:
print("Early stopping rounds:",
      booking_model_v2.get_params().get("early_stopping_rounds"))

print("Best iteration:",
      booking_model_v2.best_iteration)

print("Best validation log loss:",
      booking_model_v2.best_score)

print("Total boosting rounds:",
      booking_model_v2.n_estimators)

In [ ]:
from xgboost import XGBClassifier

improved_booking_model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=5,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

improved_booking_model.fit(
    X_train_booking_v2,
    y_train_booking,
    eval_set=[(X_val_booking_v2, y_val_booking)],
    verbose=100
)

print("Improved booking-time model training completed!")
print("Best iteration:", improved_booking_model.best_iteration)
print("Best validation log loss:", improved_booking_model.best_score)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

y_pred_improved = improved_booking_model.predict(
    X_val_booking_v2
)

y_prob_improved = improved_booking_model.predict_proba(
    X_val_booking_v2
)[:, 1]

print("IMPROVED BOOKING-TIME XGBOOST RESULTS")

print("Accuracy:",
      round(accuracy_score(y_val_booking, y_pred_improved) * 100, 2), "%")

print("Precision:",
      round(precision_score(y_val_booking, y_pred_improved) * 100, 2), "%")

print("Recall:",
      round(recall_score(y_val_booking, y_pred_improved) * 100, 2), "%")

print("F1 Score:",
      round(f1_score(y_val_booking, y_pred_improved) * 100, 2), "%")

print("ROC-AUC:",
      round(roc_auc_score(y_val_booking, y_prob_improved), 4))

print("\nClassification Report:")
print(classification_report(y_val_booking, y_pred_improved))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_booking, y_pred_improved))

In [ ]:

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from google.colab import files

# Compare all four main models
candidates = [
    {
        "name": "Original Tuned XGBoost",
        "model": tuned_model,
        "X": X_val,
        "y": y_val,
        "columns": list(X_tr.columns),
        "type": "Original"
    },
    {
        "name": "Booking XGBoost (36 features)",
        "model": booking_model,
        "X": X_val_booking,
        "y": y_val_booking,
        "columns": list(X_train_booking.columns),
        "type": "Booking-time candidate"
    },
    {
        "name": "Strict Booking XGBoost V2",
        "model": booking_model_v2,
        "X": X_val_booking_v2,
        "y": y_val_booking,
        "columns": list(X_train_booking_v2.columns),
        "type": "Strict booking-time candidate"
    },
    {
        "name": "Improved Booking XGBoost",
        "model": improved_booking_model,
        "X": X_val_booking_v2,
        "y": y_val_booking,
        "columns": list(X_train_booking_v2.columns),
        "type": "Strict booking-time candidate"
    }
]

results = []

for item in candidates:
    predictions = item["model"].predict(item["X"])
    probabilities = item["model"].predict_proba(item["X"])[:, 1]

    results.append({
        "Model": item["name"],
        "Type": item["type"],
        "Accuracy": round(accuracy_score(item["y"], predictions) * 100, 2),
        "Precision": round(precision_score(item["y"], predictions) * 100, 2),
        "Recall": round(recall_score(item["y"], predictions) * 100, 2),
        "F1 Score": round(f1_score(item["y"], predictions) * 100, 2),
        "ROC-AUC": round(roc_auc_score(item["y"], probabilities), 4)
    })

results_df = pd.DataFrame(results).sort_values(
    by="Accuracy", ascending=False
)

print("FINAL MODEL COMPARISON")
display(results_df)

# Select the highest-accuracy model overall
best_overall_name = results_df.iloc[0]["Model"]
best_overall = next(
    item for item in candidates if item["name"] == best_overall_name
)

# Select the highest-accuracy booking-time candidate
booking_candidates = [
    item for item in candidates
    if item["type"] in [
        "Booking-time candidate",
        "Strict booking-time candidate"
    ]
]

booking_scores = results_df[
    results_df["Model"].isin([item["name"] for item in booking_candidates])
]

best_booking_name = booking_scores.iloc[0]["Model"]
best_booking = next(
    item for item in booking_candidates
    if item["name"] == best_booking_name
)

# Save model packages with their required feature columns
joblib.dump({
    "model": best_overall["model"],
    "feature_columns": best_overall["columns"],
    "model_name": best_overall["name"],
    "validation_accuracy": float(
        results_df.iloc[0]["Accuracy"]
    )
}, "best_overall_model.joblib")

joblib.dump({
    "model": best_booking["model"],
    "feature_columns": best_booking["columns"],
    "model_name": best_booking["name"],
    "validation_accuracy": float(
        booking_scores.iloc[0]["Accuracy"]
    )
}, "best_booking_time_model.joblib")

# Save comparison report
results_df.to_csv("model_comparison_report.csv", index=False)

print("\nBEST OVERALL MODEL:", best_overall_name)
print("BEST BOOKING-TIME CANDIDATE:", best_booking_name)

# Download files
files.download("best_overall_model.joblib")
files.download("best_booking_time_model.joblib")
files.download("model_comparison_report.csv")
